In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("/content/zomato_data.csv")

In [ ]:
clean = df.copy()

**1.Rename columns**

In [ ]:
clean = clean.rename(columns={
    "approx_cost(for two people)": "cost_for_two",
    "listed_in(type)": "listing_type",
    "listed_in(city)": "listing_area",
})
clean.columns

Index(['url', 'address', 'name', 'online_order', 'book_table', 'rate', 'votes',
       'phone', 'location', 'rest_type', 'dish_liked', 'cuisines',
       'cost_for_two', 'listing_type', 'listing_area'],
      dtype='object')

**2.Rating**

In [ ]:
clean["is_new_restaurant"] = clean["rate"] == "NEW"

clean["rating"] = (clean["rate"]
                   .replace({"NEW": np.nan, "-": np.nan})
                   .str.replace("/5", "", regex=False)
                   .str.strip())
clean["rating"] = pd.to_numeric(clean["rating"])
clean = clean.drop(columns="rate")

clean["rating"].describe()

,rating
count,41665.000000
mean,3.700449
std,0.440513
min,1.800000
25%,3.400000
50%,3.700000
75%,4.000000
max,4.900000


**3.Cost**

In [ ]:
clean["cost_for_two"] = pd.to_numeric(
    clean["cost_for_two"].str.replace(",", "", regex=False))
clean["cost_for_two"].describe()

,cost_for_two
count,51371.000000
mean,555.431566
std,438.850728
min,40.000000
25%,300.000000
50%,400.000000
75%,650.000000
max,6000.000000


**4.Fix broken characters**

In [ ]:
def fix_encoding(text):
    if not isinstance(text, str):
        return text
    for _ in range(20):
        try:
            fixed = text.encode("latin-1").decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            break
        if fixed == text:
            break
        text = fixed
    return text

for col in ["name", "address", "location", "rest_type", "dish_liked", "cuisines"]:
    clean[col] = clean[col].map(fix_encoding)

clean["name"].str.contains("Ã|Â").sum()

np.int64(0)

**5.Small fixes**

In [ ]:
text_cols = ["name", "address", "location", "rest_type", "cuisines", "dish_liked"]
for col in text_cols:
    clean[col] = clean[col].str.strip()

clean["online_order"] = clean["online_order"].map({"Yes": True, "No": False})
clean["book_table"] = clean["book_table"].map({"Yes": True, "No": False})
clean["phone"] = clean["phone"].str.replace(r"\r\n", " | ", regex=True)

**6.Missing values**

In [ ]:
clean["location"] = clean["location"].fillna(clean["listing_area"])
clean["rest_type"] = clean["rest_type"].fillna("Unknown")
clean["cuisines"] = clean["cuisines"].fillna("Unknown")
clean["dish_liked"] = clean["dish_liked"].fillna("Not available")

clean["cost_for_two"] = clean["cost_for_two"].fillna(
    clean.groupby("location")["cost_for_two"].transform("median"))
clean["cost_for_two"] = clean["cost_for_two"].fillna(clean["cost_for_two"].median())

clean.isnull().sum()

,0
url,0
address,0
name,0
online_order,0
book_table,0
votes,0
phone,1208
location,0
rest_type,0
dish_liked,0


**7.Remove duplicates**

In [ ]:
listings = (clean[["name", "address", "listing_type", "listing_area"]]
            .drop_duplicates()
            .reset_index(drop=True))

restaurants = (clean.sort_values("votes", ascending=False)
               .drop_duplicates(subset=["name", "address"], keep="first")
               .drop(columns=["url", "listing_type", "listing_area"])
               .reset_index(drop=True))

print(restaurants.shape)
restaurants.duplicated(["name", "address"]).sum()

(12464, 13)


np.int64(0)

**8.Outliers and new columns**

In [ ]:
q1, q3 = restaurants["votes"].quantile([0.25, 0.75])
restaurants["is_high_votes_outlier"] = restaurants["votes"] > q3 + 1.5 * (q3 - q1)

restaurants["price_category"] = pd.cut(
    restaurants["cost_for_two"], bins=[0, 300, 600, 1000, np.inf],
    labels=["Budget", "Mid", "Premium", "Luxury"])

restaurants["rating_category"] = pd.cut(
    restaurants["rating"], bins=[0, 3.0, 3.5, 4.0, 5.0],
    labels=["Poor", "Average", "Good", "Excellent"])

restaurants["primary_cuisine"] = restaurants["cuisines"].str.split(",").str[0].str.strip()
restaurants["cuisine_count"] = restaurants["cuisines"].str.split(",").str.len()
restaurants["primary_rest_type"] = restaurants["rest_type"].str.split(",").str[0].str.strip()

**9.Cuisine table**

In [ ]:
cuisine_table = (restaurants[["name", "address", "cuisines"]]
                 .assign(cuisine=lambda d: d["cuisines"].str.split(","))
                 .explode("cuisine")
                 .assign(cuisine=lambda d: d["cuisine"].str.strip())
                 .drop(columns="cuisines"))
cuisine_table.head()

,name,address,cuisine
0,Byg Brewski Brewing Company,"Behind MK Retail, Sarjapur Road, Bangalore",Continental
0,Byg Brewski Brewing Company,"Behind MK Retail, Sarjapur Road, Bangalore",North Indian
0,Byg Brewski Brewing Company,"Behind MK Retail, Sarjapur Road, Bangalore",Italian
0,Byg Brewski Brewing Company,"Behind MK Retail, Sarjapur Road, Bangalore",South Indian
0,Byg Brewski Brewing Company,"Behind MK Retail, Sarjapur Road, Bangalore",Finger Food


**10.Validation**

In [ ]:
assert restaurants["rating"].dropna().between(0, 5).all()
assert (restaurants["cost_for_two"] > 0).all()
assert (restaurants["votes"] >= 0).all()
assert restaurants.duplicated(["name", "address"]).sum() == 0
print("All checks passed")

# Extra checks to think about
print(restaurants[(restaurants["votes"] == 0) & restaurants["rating"].notna()].shape)
print(restaurants[(restaurants["votes"] > 100) & restaurants["rating"].isna()].shape)
sorted(restaurants["location"].unique())

All checks passed
(0, 19)
(13, 19)


['BTM',
 'Banashankari',
 'Banaswadi',
 'Bannerghatta Road',
 'Basavanagudi',
 'Basaveshwara Nagar',
 'Bellandur',
 'Bommanahalli',
 'Brigade Road',
 'Brookefield',
 'CV Raman Nagar',
 'Central Bangalore',
 'Church Street',
 'City Market',
 'Commercial Street',
 'Cunningham Road',
 'Domlur',
 'East Bangalore',
 'Ejipura',
 'Electronic City',
 'Frazer Town',
 'HBR Layout',
 'HSR',
 'Hebbal',
 'Hennur',
 'Hosur Road',
 'ITPL Main Road, Whitefield',
 'Indiranagar',
 'Infantry Road',
 'JP Nagar',
 'Jakkur',
 'Jalahalli',
 'Jayanagar',
 'Jeevan Bhima Nagar',
 'KR Puram',
 'Kaggadasapura',
 'Kalyan Nagar',
 'Kammanahalli',
 'Kanakapura Road',
 'Kengeri',
 'Koramangala',
 'Koramangala 1st Block',
 'Koramangala 2nd Block',
 'Koramangala 3rd Block',
 'Koramangala 4th Block',
 'Koramangala 5th Block',
 'Koramangala 6th Block',
 'Koramangala 7th Block',
 'Koramangala 8th Block',
 'Kumaraswamy Layout',
 'Langford Town',
 'Lavelle Road',
 'MG Road',
 'Magadi Road',
 'Majestic',
 'Malleshwaram',
 'M

**11.Saving the csv files**

In [ ]:
restaurants.to_csv("zomato_clean_restaurants.csv", index=False)
listings.to_csv("zomato_listings.csv", index=False)
cuisine_table.to_csv("zomato_cuisines.csv", index=False)

In [ ]:
print(restaurants.shape)
print(listings.shape)
print(cuisine_table.shape)
print(restaurants.isnull().sum())
restaurants.duplicated(["name", "address"]).sum()

(12464, 19)
(51628, 4)
(28722, 3)
address                     0
name                        0
online_order                0
book_table                  0
votes                       0
phone                     308
location                    0
rest_type                   0
dish_liked                  0
cuisines                    0
cost_for_two                0
is_new_restaurant           0
rating                   3022
is_high_votes_outlier       0
price_category              0
rating_category          3022
primary_cuisine             0
cuisine_count               0
primary_rest_type           0
dtype: int64


np.int64(0)

In [ ]:
# 1. Give each restaurant an ID
restaurants = restaurants.reset_index(drop=True)
restaurants.insert(0, "restaurant_id", restaurants.index + 1)

# 2. Attach that ID to the other two tables
key = restaurants[["restaurant_id", "name", "address"]]

listings_db = (listings.merge(key, on=["name", "address"])
               [["restaurant_id", "listing_type", "listing_area"]])

cuisines_db = (cuisine_table.merge(key, on=["name", "address"])
               [["restaurant_id", "cuisine"]])

print(restaurants.shape, listings_db.shape, cuisines_db.shape)

(12464, 20) (51628, 3) (28722, 2)


In [ ]:
restaurants.to_csv("restaurants.csv", index=False)
listings_db.to_csv("listings.csv", index=False)
cuisines_db.to_csv("cuisines.csv", index=False)